#### Imports

In [ ]:
# Imports
import os
import sys
import json
from tqdm import tqdm
from PIL import Image
import supervision as sv
from rfdetr import RFDETRMedium
from supervision.metrics import MeanAveragePrecision

#### Path Configurations

In [2]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
data_directory = os.path.join(project_root , "data" , "detection")
dataset_name = "coco_football_players_detection_v11"
dataset_path = os.path.join(data_directory , dataset_name)

# Model Name and Weights
base_model_name = "rfdetr_m"
model_name = "04-05-2026_08-51_rfdetr_m"
model_directory = os.path.join(project_root , "models" , "detection", model_name)
full_model_weights_path = os.path.join(model_directory, "checkpoint_best_ema.pth")

#### Model Setup

In [3]:
detection_model = RFDETRMedium(pretrain_weights=full_model_weights_path, num_classes = 4)
detection_model.optimize_for_inference()

[2026-05-04 08:27:43] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-04 08:27:43] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


#### Loading Dataset

In [4]:
test_dataset = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset_path}/test",
    annotations_path=f"{dataset_path}/test/_annotations.coco.json",
)

#### Model Metrics

In [5]:
# Gathering Model Statistics
targets = []
predictions = []


for path, image, annotations in tqdm(test_dataset):
    image = Image.open(path)
    detections = detection_model.predict(image, threshold=0)

    targets.append(annotations)
    predictions.append(detections)

100%|██████████| 25/25 [00:07<00:00,  3.42it/s]


#### Per-class and Overall Stats

In [6]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

class_names = test_dataset.classes

per_class_metrics = {}
for i, name in enumerate(class_names):
    per_class_metrics[name] = {
        "mAP50-95": round(float(map_result.ap_per_class[i].mean()), 4),
        "mAP50":    round(float(map_result.ap_per_class[i, 0]), 4),
    }

overall_metrics = {
    "mAP50-95": round(float(map_result.map50_95), 4),
    "mAP50":    round(float(map_result.map50), 4),
}

#### Full Metrics

In [ ]:
# Full Metrics
metrics_to_save = {
    'model_name': model_name,
    'pretrained_base_model': base_model_name,
    'evaluation_dataset': dataset_name,
    'imgsz': ,
    'overall_metrics': overall_metrics,
    'per_class_overall_metrics':per_class_metrics,
    }

#### Saving JSON Metrics

In [8]:
full_model_eval_results_path = os.path.join(model_directory, f'{model_name}_{dataset_name}_results.json')
with open(full_model_eval_results_path, "w") as f:
    json.dump(metrics_to_save, f, indent=4)